# 02 Small Model Training, Quantization and Conversion

## Setup and imports

In [ ]:
import sys

COLAB = "google.colab" in sys.modules

# Environment Setup
if COLAB:
    print("Running in Google Colab. Setting up environment...")
    BASE_DIR = './'  # In Colab, everything stays inside /content/

    import gdown
    requirements_ID = '1XHtGbTN-NEHnvAyM7ufYi4mw0K2lN9NN'
    gdown.download(id=requirements_ID, output='requirements.txt', quiet=False)
    %pip install -r "requirements.txt"
else:
    print("Running locally. Assuming files are already present.")
    BASE_DIR = '../' # Locally, everything is one level up from /src

    requirements_path = f"{BASE_DIR}requirements.txt"
    %pip install -r "{requirements_path}"

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import tensorflow as tf
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Conv2D, Flatten, Dense, ReLU, Input, Dropout, BatchNormalization, ZeroPadding2D
from tf_keras.optimizers import Adam
from tf_keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from quantizeml import load_model
from quantizeml.models import quantize, QuantizationParams
from quantizeml.models.quantize import dump_config
import akida
import cnn2snn
from cnn2snn import set_akida_version, AkidaVersion, convert

In [ ]:
# Force TensorFlow to use the legacy Keras 2 backend
# This is strictly required because Brainchip's cnn2snn converter crashes with Keras 3
os.environ["TF_USE_LEGACY_KERAS"] = "1"

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras module: {tf_keras.__name__}")
print(f"Keras version: {tf_keras.__version__}")

## Dataset download and load
We directly download and load the preprocessed, hand-oriented filtered (small) of our dataset, bypassing the raw data processing steps

In [ ]:
# Dataset download
dataset_dir = os.path.join(BASE_DIR, 'data/processed_data/filtered')
dataset_path = os.path.join(dataset_dir, 'X_watch_20Hz_scaled_small.npy')
labels_path = os.path.join(dataset_dir, 'y_watch_labels_small.npy')

if not os.path.exists(dataset_path) or not os.path.exists(labels_path):
    dataset_ID = '1ejv0RQpc_VE9m-iHuo2ZppY19JPWBUWY'
    labels_ID = '16TnsGVSvc_FJ33HyS2ASoQKqN4r20e8W'

    print(f"Dataset or labels not found in {dataset_dir}. Downloading...")
    os.makedirs(dataset_dir, exist_ok=True)
        
    gdown.download(id=dataset_ID, output=dataset_path, quiet=False)
    gdown.download(id=labels_ID, output=labels_path, quiet=False)
    print("Download complete!")
else:
    print(f"Dataset and labels already exist in {dataset_dir}. Skipping download.")

In [ ]:
# Dataset load
try:
    X_data = np.load(dataset_path)
    y_labels = np.load(labels_path).astype(np.int32)

    print(f"Dataset loaded successfully from: {dataset_path}")
    print(f"Labels loaded successfully from: {labels_path}")

    print(f"\nX_data shape: {X_data.shape}")
    print(f"y_labels shape: {y_labels.shape}, dtype: {y_labels.dtype}")

except FileNotFoundError:
    print(f"Error: One or both files were not found.")
except Exception as e:
    print(f"An error occurred: {e}")

## Dataset split and preprocess

In [ ]:
def split_and_save_dataset(X, y, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, train_size=train_size, random_state=random_state, stratify=y
    )

    relative_val_size = val_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, train_size=relative_val_size, random_state=random_state, stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_and_save_dataset(X_data, y_labels, train_size=0.7, val_size=0.15, test_size=0.15)

# Remap the labels into a continuous range [0, N-1]
unique_labels = np.unique(y_train)
label_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_labels)}

def apply_mapping(labels, mapping):
    return np.array([mapping[label] for label in labels], dtype=np.int32)

y_train = apply_mapping(y_train, label_mapping)
y_val = apply_mapping(y_val, label_mapping)
y_test = apply_mapping(y_test, label_mapping)

num_classes = len(unique_labels)
    
print(f"Number of unique classes detected: {num_classes}")
print(f"Applied mapping: {label_mapping}")
print(f"X_train dimensions: {X_train.shape}, y_train: {y_train.shape}")

In [ ]:
X_train_reshaped = X_train.reshape(-1, 40, 6, 1)
X_val_reshaped = X_val.reshape(-1, 40, 6, 1)
X_test_reshaped = X_test.reshape(-1, 40, 6, 1)

# 41x9 for hardware symmetric padding
pad_config = ((0,0), (0,1), (1,2), (0,0))
X_train_padded = np.pad(X_train_reshaped, pad_width=pad_config, mode='constant', constant_values=0)
X_val_padded = np.pad(X_val_reshaped, pad_width=pad_config, mode='constant', constant_values=0)
X_test_padded = np.pad(X_test_reshaped, pad_width=pad_config, mode='constant', constant_values=0)

x_min = np.min(X_train_padded)
x_max = np.max(X_train_padded)

# Range [0.0, 1.0] in float32 (for input_quantizer)
X_train = ((X_train_padded - x_min) / (x_max - x_min + 1e-8)).astype(np.float32)
X_val = ((X_val_padded - x_min) / (x_max - x_min + 1e-8)).astype(np.float32)
X_test = ((X_test_padded - x_min) / (x_max - x_min + 1e-8)).astype(np.float32)

## Model (small)
We call this the "small" model because it is trained on the filtered subset of the data (small).

In [ ]:
def create_akida_compatible_baseline(num_classes):
    model = Sequential([
        Input(shape=(41, 9, 1)),

        # output (21, 5, 32)
        Conv2D(filters=32, kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=6),

        # output (11, 3, 64)
        Conv2D(filters=64, kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=6),

        # output (6, 2, 128)
        Conv2D(filters=128, kernel_size=(3, 3), strides=(2, 2), padding='same'),
        ReLU(max_value=6),

        # output (6,2,48)
        Conv2D(filters=48, kernel_size=(1, 1), strides=(1, 1), padding='same'),
        ReLU(max_value=6),

        # output (576)
        Flatten(),

        Dense(64),
        ReLU(max_value=6),
        Dropout(0.45),

        Dense(32),
        ReLU(max_value=6),
        Dropout(0.45),

        Dense(num_classes)
    ])
    return model

baseline_model = create_akida_compatible_baseline(num_classes)
LR = 5e-4

baseline_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss=tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

baseline_model.summary()

### Training Loop

In [ ]:
# Create the directory to save models if it does not exist
models_dir = os.path.join(BASE_DIR, 'models')
checkpoints_dir = os.path.join(models_dir, 'checkpoint')
os.makedirs(checkpoints_dir, exist_ok=True)

model_path = os.path.join(checkpoints_dir, 'final_model_small.h5')

# Definition of Callbacks
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=30,
    restore_best_weights=True
)

# Reduce the LR if the val_accuracy does not improve for 15 epochs
reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=15,
    min_lr=1e-6,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath=model_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Training (Training Loop)
EPOCHS = 500
BATCH_SIZE = 128

In [ ]:
print("Starting training...")
history = baseline_model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, model_checkpoint, reduce_lr],
    verbose=1
)

### Evaluation on Test Set

In [ ]:
print("--- Evaluation on Test Set ---")
test_loss, test_accuracy = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Visualization of the results
plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

### Saving the Model

In [ ]:
# Define the path
final_model_path = os.path.join(models_dir, 'final_model_small.h5')
baseline_model.save(final_model_path)

print(f"Final model successfully saved in:\n{final_model_path}")

## Quantization with QuantizeML

In [ ]:
# Loading the model with optimal weights
best_model_path = os.path.join(models_dir, 'final_model_small.h5')
baseline_model.load_weights(best_model_path)
print(f"Float model weights loaded from: {best_model_path}")

# Evaluation on the Test Set
print("\n--- Evaluation on the Test Set (baseline model) ---")
test_loss, test_accuracy = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

In [ ]:
num_calibration_samples = 2048
indices = np.random.choice(len(X_train), num_calibration_samples, replace=False)
calibration_data = X_train[indices]

qparams = QuantizationParams(
    input_weight_bits=8,
    weight_bits=4,
    activation_bits=4,
)

with set_akida_version(AkidaVersion.v1):
    quantized_model = quantize(
        baseline_model,
        qparams=qparams,
        samples=calibration_data
    )

print("\n--- Quantized Model Structure ---")
quantized_model.summary()

quantized_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("\n--- Quantized Model Evaluation ---")
q_test_loss, q_test_accuracy = quantized_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (Quantized): {q_test_loss:.4f}")
print(f"Test Accuracy (Quantized): {q_test_accuracy*100:.2f}%")

### Fine Tuning Quantized Model


In [ ]:
print("--- Quantized Model Fine-Tuning (QAT) Alignment Strategy ---")

# Google Drive path
qat_model_path = os.path.join(checkpoints_dir, 'final_model_small_quantized.h5')
os.makedirs(os.path.dirname(qat_model_path), exist_ok=True)

qat_checkpoint = ModelCheckpoint(
    filepath=qat_model_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

qat_early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

qat_reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

quantized_model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss=tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Training
history_qat = quantized_model.fit(
    X_train, y_train,
    epochs=2,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[qat_checkpoint, qat_early_stopping, qat_reduce_lr]
)

print(f"Fine-tuning completed and best weights directly saved in: {qat_model_path}")

### Evaluation on Test Set

In [ ]:
# Evaluation on the Test Set
print("--- Evaluation on the Test Set ---")
test_loss, test_accuracy = quantized_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

### Saving the Finetuned Quantized Model

In [ ]:
# Define the path
qat_model_path = os.path.join(models_dir, 'final_model_small_quantized.h5')
quantized_model.save_weights(qat_model_path)
print(f"Fine-tuned weights saved successfully to: {qat_model_path}")

## Convertion to Akida SNN

In [ ]:
# Load the best QAT weights into the quantized model
qat_model_path = os.path.join(models_dir, 'final_model_small_quantized.h5')
quantized_model.load_weights(qat_model_path)
print(f"QAT weights loaded from: {qat_model_path}")

# Verify that the weights are the correct ones
_, acc_check = quantized_model.evaluate(X_test, y_test, verbose=0)
print(f"Quantized accuracy with QAT weights (pre-conversion): {acc_check*100:.2f}%")

# Conversion
with set_akida_version(AkidaVersion.v1):
    akida_model = cnn2snn.convert(quantized_model)

print("Conversion completed.")

# Evaluation
preds = akida_model.predict(X_test)
akida_preds = np.argmax(preds, axis=-1).flatten()
akida_accuracy = np.mean(akida_preds == y_test)
print(f"\nAkida Accuracy: {akida_accuracy * 100:.2f}%")

# HW Mapping
device = akida.AKD1000()
akida_model.map(device=device)
akida_model.summary()

### Saving the Finetuned Quantized Model

In [ ]:
akida_fb_path = os.path.join(models_dir, 'final_model_small_quantized_akida.fb')
akida_model.save(akida_fb_path)

print(f"Akida SNN model saved successfully in .fb format: {akida_fb_path}")